# MRI super-resolution versus interpolation

Use a small central T1 crop, reduce its through-plane sampling, then compare
interpolation with the pretrained SIQ deep back-projection model. This is an
illustrative reconstruction experiment, not a simulation of all effects of
lower-resolution MRI acquisition. **Resources:** downloads a sample T1 and
converted model weights; full-volume super-resolution can require much more
memory than this crop. Numerical agreement with the original SIQ implementation
is not established by this notebook.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

## Load the sample MRI or choose a local image

In [ ]:
from pathlib import Path

# Set an absolute path to use your own scalar 3-D T1 MRI instead.
t1_path = None
if t1_path is None:
    t1_path = antstorch.get_antstorch_data("mprage_hippmapp3r")
t1 = ants.image_read(str(t1_path)).clone("float")
assert t1.dimension == 3 and not t1.has_components
assert np.isfinite(t1.numpy()).all()
print("Input:", t1_path, "shape:", t1.shape, "spacing:", t1.spacing)

In [ ]:
crop_size = np.array([32, 32, 32])
if np.any(np.array(t1.shape) < crop_size):
    raise ValueError("Use a T1 image with at least 32 voxels per axis for this example.")
start = (np.array(t1.shape) - crop_size) // 2
high_resolution = ants.crop_indices(t1, start.tolist(), (start+crop_size).tolist())
# Gaussian prefilter before decimation reduces through-plane aliasing.
smoothed = ants.smooth_image(high_resolution, (0., 0., 0.8), sigma_in_physical_coordinates=False)
low_array = smoothed.numpy()[:, :, ::2].copy()
low_spacing = np.array(high_resolution.spacing) * np.array([1, 1, 2])
low_resolution = ants.from_numpy(low_array, spacing=tuple(low_spacing),
                                 origin=high_resolution.origin, direction=high_resolution.direction)
interpolated = ants.resample_image_to_target(low_resolution, high_resolution, interp_type="linear")
print("Low-resolution shape/spacing:", low_resolution.shape, low_resolution.spacing)

## Apply pretrained super-resolution
Expansion factors are in ANTs physical axis order `(x, y, z)`. The `(1, 1, 2)`
VGG-feature model doubles z samples and halves z spacing. It does not create
newly measured anatomy. Keep the default intensity matching explicit when
comparing numerical errors with an interpolation baseline.

In [ ]:
with time_step('MRI super-resolution (super_resolved)'):
    super_resolved = antstorch.mri_super_resolution(
        low_resolution, expansion_factor=(1, 1, 2), feature="vgg",
        poly_order="hist", device=device, verbose=True,
    )
assert super_resolved.shape == high_resolution.shape
assert np.isfinite(super_resolved.numpy()).all()
for attribute in ("spacing", "origin", "direction"):
    np.testing.assert_allclose(getattr(super_resolved, attribute), getattr(high_resolution, attribute))
# Match the interpolation baseline to the same low-resolution intensity reference.
interpolation_matched = ants.histogram_match_image(interpolated, low_resolution)
reference = high_resolution.numpy()
baseline = interpolation_matched.numpy()
prediction = super_resolved.numpy()
for name, values in [("Interpolation (histogram matched)", baseline), ("Super-resolution", prediction)]:
    rmse = float(np.sqrt(np.mean((values-reference)**2)))
    print(f"{name} RMSE against original crop: {rmse:.3f}")

In [ ]:
# A sagittal slice shows the z direction affected by this model.
slice_index = high_resolution.shape[0] // 2
fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
views = [reference[slice_index], low_array[slice_index], baseline[slice_index], prediction[slice_index]]
titles = ["Original crop", "Reduced z sampling", "Interpolation", "Super-resolution"]
for ax, values, title in zip(axes, views, titles):
    ax.imshow(values.T, origin="lower", cmap="gray", aspect="auto",
              vmin=reference.min(), vmax=reference.max())
    ax.set_title(title); ax.axis("off")
plt.show()
output_dir = Path("example_outputs/mri_super_resolution")
output_dir.mkdir(parents=True, exist_ok=True)
for name, image in {"original_crop": high_resolution, "low_resolution": low_resolution,
                    "interpolation": interpolation_matched, "super_resolution": super_resolved}.items():
    ants.image_write(image, str(output_dir / f"{name}.nii.gz"))
print("Outputs (matching files are overwritten):", output_dir.resolve())

This example deliberately makes no assertion that the learned model must
outperform interpolation. Inspect small structures, ringing and synthetic
detail. Histogram matching changes intensity statistics, so these RMSE values
describe this particular comparison rather than a general accuracy result.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")